# Setup

## Imports

In [55]:
# Cell 1 - Imports and paths

import math
from pathlib import Path
from itertools import combinations
from collections import defaultdict

import numpy as np
import pandas as pd


## Settings

In [56]:
LOG_RUN = False

TARGET_PATH = "data/target.sdf"
BOX_SIZE = 14
DIVISIONS = 2

NUM_FRAGMENTS = 4
BONDING_FRAGMENTS = [(1, 2), (1, 3), (2, 4)]
# Remove GNINA capping heavy atoms before final ligand assembly.
# Atom indices are 1-based and local to each fragment SDF.
FRAGMENT_CAP_ATOMS = {
    2: (6, 11),
    3: (2,),
    4: (3,),
}
# Bond anchors for decapped fragments (1-based local atom indices).
BOND_ANCHORS = {
    (1, 2): (13, 9),
    (1, 3): (10, 1),
    (2, 4): (5, 2),
}
NUM_PAIRS = int(NUM_FRAGMENTS * (NUM_FRAGMENTS - 1) / 2)
NUM_BONDING = len(BONDING_FRAGMENTS)

COMPRESSION = False
Q_FRAGMENT = 1
Q_CLASH = 1
Q_BOND = 1

ENERGY_CUTOFF = 500
SEPARATOR = 10

SAMPLER = "SA"
RUN_TAG = f"{SAMPLER}_{BOX_SIZE}_{DIVISIONS}"
LAMBDA_FRAGMENT = 100
LAMBDA_CLASH = 0
LAMBDA_BOND = 0
HARD_PENALTY = 10 * (LAMBDA_FRAGMENT * NUM_FRAGMENTS + LAMBDA_CLASH * NUM_PAIRS + LAMBDA_BOND * NUM_BONDING)

RMSD_CRITERION = 2.0


## DQM Preprocessing

In [57]:
# Cell 2 - Load raw data

fragments = {
    i: pd.read_csv(f"data/raw_{BOX_SIZE}_{DIVISIONS}/fragment_{i}_raw.csv").copy()
    for i in range(1, NUM_FRAGMENTS + 1)
}

clashes = {
    (i, j): pd.read_csv(f"data/raw_{BOX_SIZE}_{DIVISIONS}/clash_{i}_{j}_raw.csv").copy()
    for i, j in combinations(range(1, NUM_FRAGMENTS + 1), 2)
}

bond_pairs = []
for i, j in combinations(range(1, NUM_FRAGMENTS + 1), 2):
    bond_path = Path(f"data/raw_{BOX_SIZE}_{DIVISIONS}/bond_{i}_{j}_raw.csv")
    if bond_path.exists():
        bond_pairs.append((i, j))

bonds = {
    (i, j): pd.read_csv(f"data/raw_{BOX_SIZE}_{DIVISIONS}/bond_{i}_{j}_raw.csv").copy()
    for i, j in bond_pairs
}

# Keep notebook settings synchronized with what actually exists on disk
BONDING_FRAGMENTS = bond_pairs
NUM_BONDING = len(BONDING_FRAGMENTS)

missing_anchor_pairs = [pair for pair in BONDING_FRAGMENTS if pair not in BOND_ANCHORS]
if missing_anchor_pairs:
    raise ValueError(
        "Missing SDF anchor definitions for bonding pairs: "
        f"{missing_anchor_pairs}. Extend BOND_ANCHORS in notebook settings."
    )

print("Loaded fragment tables:")
for i, df in fragments.items():
    print(f"  Fragment {i}: {len(df)} rows")

print("\nLoaded clash tables:")
for pair, df in clashes.items():
    print(f"  Clash {pair}: {len(df)} rows")

print("\nLoaded bond tables:")
for pair, df in bonds.items():
    print(f"  Bond {pair}: {len(df)} rows")


Loaded fragment tables:
  Fragment 1: 452 rows
  Fragment 2: 449 rows
  Fragment 3: 516 rows
  Fragment 4: 440 rows

Loaded clash tables:
  Clash (1, 2): 202948 rows
  Clash (1, 3): 233232 rows
  Clash (1, 4): 198880 rows
  Clash (2, 3): 231684 rows
  Clash (2, 4): 197560 rows
  Clash (3, 4): 227040 rows

Loaded bond tables:
  Bond (1, 2): 202948 rows
  Bond (1, 3): 233232 rows
  Bond (2, 4): 197560 rows


In [58]:
# Cell 3 - Validate schema and basic cleanup

# Expected columns
fragment_cols = {"f", "p", "energy"}
pair_cols = {"f1", "f2", "p1", "p2", "energy"}

for i, df in fragments.items():
    if set(df.columns) != fragment_cols:
        raise ValueError(f"Fragment {i} columns mismatch: {df.columns}")
    fragments[i] = df.copy()
    fragments[i]["f"] = fragments[i]["f"].astype(int)
    fragments[i]["p"] = fragments[i]["p"].astype(int)
    fragments[i]["energy"] = fragments[i]["energy"].astype(float)

for pair, df in clashes.items():
    if set(df.columns) != pair_cols:
        raise ValueError(f"Clash {pair} columns mismatch: {df.columns}")
    clashes[pair] = df.copy()
    clashes[pair][["f1", "f2", "p1", "p2"]] = clashes[pair][["f1", "f2", "p1", "p2"]].astype(int)
    clashes[pair]["energy"] = clashes[pair]["energy"].astype(float)

for pair, df in bonds.items():
    if set(df.columns) != pair_cols:
        raise ValueError(f"Bond {pair} columns mismatch: {df.columns}")
    bonds[pair] = df.copy()
    bonds[pair][["f1", "f2", "p1", "p2"]] = bonds[pair][["f1", "f2", "p1", "p2"]].astype(int)
    bonds[pair]["energy"] = bonds[pair]["energy"].astype(float)

print("Schema validated.")


Schema validated.


In [59]:
# Cell 4 - Initialize explicit invalid flags (do NOT overwrite raw energy)

for i, df in fragments.items():
    df["is_invalid"] = False

for pair, df in clashes.items():
    df["is_invalid"] = df["energy"] > ENERGY_CUTOFF

for pair, df in bonds.items():
    df["is_invalid"] = df["energy"] > ENERGY_CUTOFF

print("Initialized invalid flags using ENERGY_CUTOFF.")


Initialized invalid flags using ENERGY_CUTOFF.


In [60]:
# Cell 5 - Symmetric invalidity propagation between clashes and bonds
# If a pair exists in both tables, invalid in one => invalid in the other

def make_pair_index(df):
    return {
        (int(r.p1), int(r.p2)): idx
        for idx, r in df[["p1", "p2"]].iterrows()
    }

for pair in BONDING_FRAGMENTS:
    if pair not in clashes or pair not in bonds:
        continue

    clash_df = clashes[pair]
    bond_df = bonds[pair]

    clash_map = make_pair_index(clash_df)
    bond_map = make_pair_index(bond_df)

    shared_keys = set(clash_map.keys()) & set(bond_map.keys())

    propagated = 0
    for key in shared_keys:
        c_idx = clash_map[key]
        b_idx = bond_map[key]

        invalid_union = bool(clash_df.at[c_idx, "is_invalid"]) or bool(bond_df.at[b_idx, "is_invalid"])
        if invalid_union:
            if not clash_df.at[c_idx, "is_invalid"]:
                clash_df.at[c_idx, "is_invalid"] = True
                propagated += 1
            if not bond_df.at[b_idx, "is_invalid"]:
                bond_df.at[b_idx, "is_invalid"] = True
                propagated += 1

    print(f"{pair}: propagated {propagated} invalid flags across clash/bond tables")


(1, 2): propagated 168950 invalid flags across clash/bond tables
(1, 3): propagated 209739 invalid flags across clash/bond tables
(2, 4): propagated 179815 invalid flags across clash/bond tables


In [61]:
# Cell 6 - Helper functions for iterative pose pruning

def pose_set_from_fragment_df(df):
    return set(df["p"].astype(int).tolist())

def build_valid_partner_maps(clashes_dict, bonds_dict):
    """
    Returns:
      valid_pair_support[(f, p)][g] = number of valid partners pose p of fragment f has in fragment g
    Counts support separately for clashes and bonds, but both contribute to survival.
    """
    valid_pair_support = defaultdict(lambda: defaultdict(int))

    # Clash support
    for (i, j), df in clashes_dict.items():
        if df.empty:
            continue
        sub = df.loc[~df["is_invalid"], ["p1", "p2"]]
        for row in sub.itertuples(index=False):
            valid_pair_support[(i, int(row.p1))][j] += 1
            valid_pair_support[(j, int(row.p2))][i] += 1

    # Bond support
    for (i, j), df in bonds_dict.items():
        if df.empty:
            continue
        sub = df.loc[~df["is_invalid"], ["p1", "p2"]]
        for row in sub.itertuples(index=False):
            valid_pair_support[(i, int(row.p1))][j] += 1
            valid_pair_support[(j, int(row.p2))][i] += 1

    return valid_pair_support


def prune_fragment_tables(fragments_dict, keep_poses):
    for f, df in fragments_dict.items():
        before = len(df)
        fragments_dict[f] = df[df["p"].isin(keep_poses[f])].copy()
        after = len(fragments_dict[f])
        if before != after:
            print(f"Fragment {f}: pruned {before - after} poses")


def prune_pair_tables(pair_dict, keep_poses):
    for (i, j), df in list(pair_dict.items()):
        before = len(df)
        mask = df["p1"].isin(keep_poses[i]) & df["p2"].isin(keep_poses[j])
        pair_dict[(i, j)] = df[mask].copy()
        after = len(pair_dict[(i, j)])
        if before != after:
            print(f"Pair {(i, j)}: removed {before - after} rows")


In [62]:
# Cell 7 - Iterative dead-pose pruning
#
# Rule:
# A pose survives only if it has at least one valid partner in every fragment
# to which it is connected by either a clash table or a bond table.
#
# This is stricter and more DQM-friendly than only checking "appears somewhere".

# Build neighbor map from available pair tables
neighbors = {f: set() for f in range(1, NUM_FRAGMENTS + 1)}

for (i, j) in clashes.keys():
    neighbors[i].add(j)
    neighbors[j].add(i)

for (i, j) in bonds.keys():
    neighbors[i].add(j)
    neighbors[j].add(i)

iteration = 0
while True:
    iteration += 1
    print(f"\n--- Pruning iteration {iteration} ---")

    keep_poses = {
        f: pose_set_from_fragment_df(df)
        for f, df in fragments.items()
    }

    support = build_valid_partner_maps(clashes, bonds)

    removed_any = False
    new_keep_poses = {}

    for f, df in fragments.items():
        surviving = set()
        for p in df["p"].astype(int):
            ok = True
            for g in neighbors[f]:
                if support[(f, p)].get(g, 0) <= 0:
                    ok = False
                    break
            if ok:
                surviving.add(int(p))

        removed = keep_poses[f] - surviving
        if removed:
            removed_any = True
            print(f"Fragment {f}: removing dead poses {sorted(list(removed))[:10]}"
                  + (" ..." if len(removed) > 10 else ""))

        new_keep_poses[f] = surviving

    if not removed_any:
        print("No more dead poses to prune.")
        break

    prune_fragment_tables(fragments, new_keep_poses)
    prune_pair_tables(clashes, new_keep_poses)
    prune_pair_tables(bonds, new_keep_poses)

    # Optional safety check
    for f, df in fragments.items():
        if df.empty:
            raise RuntimeError(f"Fragment {f} lost all poses during pruning.")



--- Pruning iteration 1 ---
Fragment 1: removing dead poses [2, 5, 9, 14, 17, 19, 22, 23, 24, 26] ...
Fragment 2: removing dead poses [6, 7, 8, 9, 14, 15, 17, 18, 19, 20] ...
Fragment 3: removing dead poses [0, 106, 121, 122, 124, 126, 127, 128, 129, 130] ...
Fragment 4: removing dead poses [9, 13, 15, 16, 17, 37, 39, 41, 78, 79] ...
Fragment 1: pruned 301 poses
Fragment 2: pruned 334 poses
Fragment 3: pruned 101 poses
Fragment 4: pruned 127 poses
Pair (1, 2): removed 185583 rows
Pair (1, 3): removed 170567 rows
Pair (1, 4): removed 151617 rows
Pair (2, 3): removed 183959 rows
Pair (2, 4): removed 161565 rows
Pair (3, 4): removed 97145 rows
Pair (1, 2): removed 185583 rows
Pair (1, 3): removed 170567 rows
Pair (2, 4): removed 161565 rows

--- Pruning iteration 2 ---
Fragment 1: removing dead poses [4, 11, 18, 20, 21, 27, 31, 32, 33, 34] ...
Fragment 2: removing dead poses [5, 11, 24, 34, 42, 63, 71, 74, 77, 104] ...
Fragment 3: removing dead poses [12, 21, 23, 35, 45, 88, 93, 94, 97, 

In [63]:
# Cell 8 - Optional extra pruning by unary fragment quality
# Keep only poses not too far above the best fragment energy
# Set USE_FRAGMENT_ENERGY_PRUNING = False to disable

USE_FRAGMENT_ENERGY_PRUNING = False
FRAGMENT_ENERGY_WINDOW = None  # e.g. 100.0

if USE_FRAGMENT_ENERGY_PRUNING:
    if FRAGMENT_ENERGY_WINDOW is None:
        raise ValueError("Set FRAGMENT_ENERGY_WINDOW when USE_FRAGMENT_ENERGY_PRUNING=True")

    keep_poses = {}
    for f, df in fragments.items():
        e_min = df["energy"].min()
        keep = set(df.loc[df["energy"] <= e_min + FRAGMENT_ENERGY_WINDOW, "p"].astype(int))
        keep_poses[f] = keep
        print(f"Fragment {f}: keeping {len(keep)}/{len(df)} poses by unary energy window")

    prune_fragment_tables(fragments, keep_poses)
    prune_pair_tables(clashes, keep_poses)
    prune_pair_tables(bonds, keep_poses)


In [64]:
# Cell 9 - Prepare scaled unary scores for fragments
# Keep raw energy untouched; create processed columns

for f, df in fragments.items():
    df = df.copy()
    e0 = df["energy"].min()
    df["energy_shifted"] = df["energy"] - e0

    max_shift = df["energy_shifted"].max()
    if max_shift > 0:
        df["score"] = df["energy_shifted"] / max_shift
    else:
        df["score"] = 0.0

    if COMPRESSION:
        # Optional monotone compression for unary terms
        df["score"] = np.log1p(Q_FRAGMENT * df["score"]) / np.log1p(Q_FRAGMENT)

    fragments[f] = df

print("Fragment unary scores prepared.")


Fragment unary scores prepared.


In [65]:
# Cell 10 - Prepare scaled pairwise scores for clashes and bonds
#
# Important:
# - valid rows get scaled scores
# - invalid rows remain explicitly marked
# - DQM export can later map invalid rows to HARD_PENALTY or forbid them conceptually

def scale_pair_table(df, cutoff, q=1.0, compression=False):
    df = df.copy()

    valid_mask = ~df["is_invalid"]
    df["energy_shifted"] = np.nan
    df["score"] = np.nan

    if valid_mask.any():
        valid_min = df.loc[valid_mask, "energy"].min()
        df.loc[valid_mask, "energy_shifted"] = df.loc[valid_mask, "energy"] - valid_min

        df.loc[valid_mask, "score"] = df.loc[valid_mask, "energy_shifted"] / cutoff

        if compression:
            df.loc[valid_mask, "score"] = np.log1p(q * df.loc[valid_mask, "score"]) / np.log1p(q)

    return df

for pair, df in clashes.items():
    clashes[pair] = scale_pair_table(
        df,
        cutoff=ENERGY_CUTOFF,
        q=Q_CLASH,
        compression=COMPRESSION
    )

for pair, df in bonds.items():
    bonds[pair] = scale_pair_table(
        df,
        cutoff=ENERGY_CUTOFF,
        q=Q_BOND,
        compression=COMPRESSION
    )

print("Pairwise scores prepared.")


Pairwise scores prepared.


In [66]:
# Cell 11 - Diagnostics summary

print("Fragments:")
for f, df in fragments.items():
    print(
        f"{f}: n={len(df)}, "
        f"min={df['score'].min():.6f}, max={df['score'].max():.6f}"
    )

print("\nClashes:")
for pair, df in clashes.items():
    valid = df.loc[~df["is_invalid"], "score"]
    invalid_count = int(df["is_invalid"].sum())
    vmin = float(valid.min()) if len(valid) else np.nan
    vmax = float(valid.max()) if len(valid) else np.nan
    print(
        f"{pair}: rows={len(df)}, valid={len(valid)}, invalid={invalid_count}, "
        f"valid_min={vmin:.6f}, valid_max={vmax:.6f}"
    )

print("\nBonds:")
for pair, df in bonds.items():
    valid = df.loc[~df["is_invalid"], "score"]
    invalid_count = int(df["is_invalid"].sum())
    vmin = float(valid.min()) if len(valid) else np.nan
    vmax = float(valid.max()) if len(valid) else np.nan
    print(
        f"{pair}: rows={len(df)}, valid={len(valid)}, invalid={invalid_count}, "
        f"valid_min={vmin:.6f}, valid_max={vmax:.6f}"
    )


Fragments:
1: n=111, min=0.000000, max=1.000000
2: n=91, min=0.000000, max=1.000000
3: n=302, min=0.000000, max=1.000000
4: n=230, min=0.000000, max=1.000000

Clashes:
(1, 2): rows=10101, valid=260, invalid=9841, valid_min=0.000000, valid_max=0.969922
(1, 3): rows=33522, valid=702, invalid=32820, valid_min=0.000000, valid_max=0.969401
(1, 4): rows=25530, valid=22312, invalid=3218, valid_min=0.000000, valid_max=1.027418
(2, 3): rows=27482, valid=24501, invalid=2981, valid_min=0.000000, valid_max=1.027125
(2, 4): rows=20930, valid=442, invalid=20488, valid_min=0.000000, valid_max=0.983156
(3, 4): rows=69460, valid=64085, invalid=5375, valid_min=0.000000, valid_max=1.008082

Bonds:
(1, 2): rows=10101, valid=260, invalid=9841, valid_min=0.000000, valid_max=0.915734
(1, 3): rows=33522, valid=702, invalid=32820, valid_min=0.000000, valid_max=0.993151
(2, 4): rows=20930, valid=442, invalid=20488, valid_min=0.000000, valid_max=0.924300


In [67]:
# Cell 12 - Build DQM-ready sparse structures
#
# Output:
#   unary_scores[f][p] = scaled unary score for pose p of fragment f
#   clash_scores[(i,j)][(p_i,p_j)] = scaled clash score for VALID pairs only
#   bond_scores[(i,j)][(p_i,p_j)] = scaled bond score for VALID pairs only
#   invalid_pairs[(i,j)] = set of invalid pose pairs
#
# Missing pair entries can later be treated as:
#   - forbidden
#   - or HARD_PENALTY in the DQM interaction matrix

unary_scores = {}
for f, df in fragments.items():
    unary_scores[f] = {
        int(row.p): float(row.score)
        for row in df[["p", "score"]].itertuples(index=False)
    }

clash_scores = {}
invalid_pairs = defaultdict(set)

for pair, df in clashes.items():
    pair_scores = {}
    for row in df[["p1", "p2", "score", "is_invalid"]].itertuples(index=False):
        key = (int(row.p1), int(row.p2))
        if bool(row.is_invalid):
            invalid_pairs[pair].add(key)
        else:
            pair_scores[key] = float(row.score)
    clash_scores[pair] = pair_scores

bond_scores = {}
for pair, df in bonds.items():
    pair_scores = {}
    for row in df[["p1", "p2", "score", "is_invalid"]].itertuples(index=False):
        key = (int(row.p1), int(row.p2))
        if bool(row.is_invalid):
            invalid_pairs[pair].add(key)
        else:
            pair_scores[key] = float(row.score)
    bond_scores[pair] = pair_scores

print("Built sparse DQM-ready structures.")


Built sparse DQM-ready structures.


In [68]:
# Cell 13 - Build full interaction tables for DQM export
#
# This cell constructs dense matrices only at the final stage.
# Invalid or missing combinations receive HARD_PENALTY.
#
# Interpretation:
#   total_pair_cost = LAMBDA_CLASH * clash + LAMBDA_BOND * bond
#
# If a pair is nonbonded, only clash contributes.
# If a pair is bonded, both clash and bond can contribute.

fragment_cases = {
    f: sorted(unary_scores[f].keys())
    for f in unary_scores
}

pair_interaction_tables = {}

for i, j in combinations(range(1, NUM_FRAGMENTS + 1), 2):
    cases_i = fragment_cases[i]
    cases_j = fragment_cases[j]

    mat = np.full((len(cases_i), len(cases_j)), HARD_PENALTY, dtype=float)

    clash_dict = clash_scores.get((i, j), {})
    bond_dict = bond_scores.get((i, j), {})

    for a, p_i in enumerate(cases_i):
        for b, p_j in enumerate(cases_j):
            key = (p_i, p_j)

            # Start with zero pair cost, then add whatever exists
            total = 0.0
            valid = True

            # Clash contribution
            if (i, j) in clashes:
                if key in invalid_pairs[(i, j)]:
                    valid = False
                elif key in clash_dict:
                    total += LAMBDA_CLASH * clash_dict[key]
                else:
                    # Missing row treated as invalid/missing support
                    valid = False

            # Bond contribution only for bonded fragment pairs
            if (i, j) in bonds:
                if key in invalid_pairs[(i, j)]:
                    valid = False
                elif key in bond_dict:
                    total += LAMBDA_BOND * bond_dict[key]
                else:
                    valid = False

            if valid:
                mat[a, b] = total

    pair_interaction_tables[(i, j)] = {
        "cases_i": cases_i,
        "cases_j": cases_j,
        "matrix": mat,
    }

print("Built dense pair interaction tables for DQM export.")


Built dense pair interaction tables for DQM export.


In [69]:
# Cell 14 - Final summary of DQM variable sizes

print("DQM variable sizes:")
for f in range(1, NUM_FRAGMENTS + 1):
    print(f"  Fragment {f}: {len(fragment_cases[f])} cases")

print("\nPair interaction matrix shapes:")
for pair, obj in pair_interaction_tables.items():
    print(f"  {pair}: {obj['matrix'].shape}")


DQM variable sizes:
  Fragment 1: 111 cases
  Fragment 2: 91 cases
  Fragment 3: 302 cases
  Fragment 4: 230 cases

Pair interaction matrix shapes:
  (1, 2): (111, 91)
  (1, 3): (111, 302)
  (1, 4): (111, 230)
  (2, 3): (91, 302)
  (2, 4): (91, 230)
  (3, 4): (302, 230)


## Build And Solve The DQM


In [70]:
# Cell 15 - Imports for DQM construction and solving

import os
import time
from getpass import getpass

import dimod
from dwave.samplers import SimulatedAnnealingSampler
from dwave.system import LeapHybridDQMSampler


In [71]:
# Cell 16 - Build the DQM

case_index_to_pose = {
    f: {case_idx: int(pose) for case_idx, pose in enumerate(fragment_cases[f])}
    for f in fragment_cases
}

pose_to_case_index = {
    f: {pose: case_idx for case_idx, pose in case_index_to_pose[f].items()}
    for f in case_index_to_pose
}

dqm = dimod.DiscreteQuadraticModel()

for f in range(1, NUM_FRAGMENTS + 1):
    poses = fragment_cases[f]
    dqm.add_variable(len(poses), label=f)
    dqm.set_linear(f, [float(unary_scores[f][pose]) for pose in poses])

for (i, j), obj in pair_interaction_tables.items():
    dqm.set_quadratic(i, j, obj["matrix"])

print("DQM variables:", dqm.num_variables())
print("Total DQM cases:", sum(dqm.num_cases(f) for f in fragment_cases))
print("DQM case interactions:", dqm.num_case_interactions())
print("DQM variable interactions:", dqm.num_variable_interactions())


DQM variables: 4
Total DQM cases: 734
DQM case interactions: 74723
DQM variable interactions: 6


In [72]:
# Cell 17 - Solve the DQM

NUM_READS = 10
NUM_SWEEPS = 100_000
DQM_SA_LAGRANGE = HARD_PENALTY


def decode_case_sample(case_sample):
    return {
        int(f): int(case_index_to_pose[f][int(case_idx)])
        for f, case_idx in case_sample.items()
    }


def decode_cqm_binary_sample(binary_sample):
    case_sample = {}
    for f, poses in fragment_cases.items():
        active = [
            case_idx
            for case_idx in range(len(poses))
            if int(round(binary_sample[(f, case_idx)])) == 1
        ]
        if len(active) != 1:
            raise ValueError(f"Fragment {f} has {len(active)} active cases in binary sample.")
        case_sample[f] = int(active[0])
    return case_sample


dqm_solution_rows = []

if SAMPLER == "SA":
    dqm_cqm = dimod.ConstrainedQuadraticModel.from_dqm(dqm)
    dqm_bqm, dqm_bqm_inverter = dimod.cqm_to_bqm(
        dqm_cqm,
        lagrange_multiplier=DQM_SA_LAGRANGE,
    )

    print("SA fallback path: DQM -> CQM -> BQM")
    print("BQM variables:", dqm_bqm.num_variables)
    print("BQM interactions:", dqm_bqm.num_interactions)

    sampler = SimulatedAnnealingSampler()
    t0 = time.perf_counter()
    dqm_raw_sampleset = sampler.sample(
        dqm_bqm,
        num_reads=NUM_READS,
        num_sweeps=NUM_SWEEPS,
    )
    SOLVING_TIME = time.perf_counter() - t0
    dqm_raw_sampleset = dqm_raw_sampleset.aggregate()

    for record in dqm_raw_sampleset.data(["sample", "num_occurrences"]):
        cqm_sample = dqm_bqm_inverter(record.sample)
        if not dqm_cqm.check_feasible(cqm_sample):
            continue

        case_sample = decode_cqm_binary_sample(cqm_sample)
        pose_sample = decode_case_sample(case_sample)

        row = {
            "solver": "SA",
            "energy": float(dqm.energy(case_sample)),
            "num_occurrences": int(record.num_occurrences),
        }
        for f in range(1, NUM_FRAGMENTS + 1):
            row[f"case_{f}"] = case_sample[f]
            row[f"pose_{f}"] = pose_sample[f]
        dqm_solution_rows.append(row)

    if not dqm_solution_rows:
        raise RuntimeError("No feasible DQM samples were recovered from the SA fallback path.")

elif SAMPLER == "DQM":
    token = os.getenv("DWAVE_API_TOKEN") or getpass("Enter D-Wave API token: ")
    sampler = LeapHybridDQMSampler(token=token)

    t0 = time.perf_counter()
    dqm_raw_sampleset = sampler.sample_dqm(dqm)
    SOLVING_TIME = time.perf_counter() - t0
    dqm_raw_sampleset = dqm_raw_sampleset.aggregate()

    for record in dqm_raw_sampleset.data(["sample", "energy", "num_occurrences"]):
        case_sample = {
            int(f): int(record.sample[f])
            for f in range(1, NUM_FRAGMENTS + 1)
        }
        pose_sample = decode_case_sample(case_sample)

        row = {
            "solver": "DQM",
            "energy": float(record.energy),
            "num_occurrences": int(record.num_occurrences),
        }
        for f in range(1, NUM_FRAGMENTS + 1):
            row[f"case_{f}"] = case_sample[f]
            row[f"pose_{f}"] = pose_sample[f]
        dqm_solution_rows.append(row)

else:
    raise NameError(f"SAMPLER name {SAMPLER} is not valid. Please use SA or DQM")

dqm_solution_frame = pd.DataFrame(dqm_solution_rows).sort_values(
    ["energy", "num_occurrences"],
    ascending=[True, False],
).reset_index(drop=True)

print(f"Solving time ({SAMPLER}): {SOLVING_TIME:.3f} s")
print("Recovered feasible solutions:", len(dqm_solution_frame))
dqm_solution_frame.head()


SA fallback path: DQM -> CQM -> BQM
BQM variables: 734
BQM interactions: 156709
Solving time (SA): 36.853 s
Recovered feasible solutions: 10


,solver,energy,num_occurrences,case_1,pose_1,case_2,pose_2,case_3,pose_3,case_4,pose_4
0,SA,0.103706,1,12,30,18,73,63,76,40,69
1,SA,0.116817,1,12,30,18,73,150,232,100,169
2,SA,0.254400,1,5,8,38,220,63,76,106,210
3,SA,0.571824,1,78,340,19,78,62,75,94,163
4,SA,0.573584,1,78,340,19,78,62,75,99,168


In [73]:
# Cell 18 - Inspect the best DQM solution

best_dqm_solution = dqm_solution_frame.iloc[0].to_dict()

best_case_sample = {
    f: int(best_dqm_solution[f"case_{f}"])
    for f in range(1, NUM_FRAGMENTS + 1)
}

best_pose_sample = {
    f: int(best_dqm_solution[f"pose_{f}"])
    for f in range(1, NUM_FRAGMENTS + 1)
}

best_unary_total = sum(
    float(unary_scores[f][best_pose_sample[f]])
    for f in range(1, NUM_FRAGMENTS + 1)
)

best_pair_rows = []
best_pair_total = 0.0

for i, j in combinations(range(1, NUM_FRAGMENTS + 1), 2):
    pair_cost = float(
        pair_interaction_tables[(i, j)]["matrix"][best_case_sample[i], best_case_sample[j]]
    )
    best_pair_total += pair_cost
    best_pair_rows.append(
        {
            "pair": (i, j),
            "pose_i": best_pose_sample[i],
            "pose_j": best_pose_sample[j],
            "pair_cost": pair_cost,
        }
    )

print("Best DQM solution")
print(f"  Solver: {best_dqm_solution['solver']}")
print(f"  Energy: {best_dqm_solution['energy']:.6f}")
print(f"  Num occurrences: {int(best_dqm_solution['num_occurrences'])}")
print(f"  Unary total: {best_unary_total:.6f}")
print(f"  Pair total: {best_pair_total:.6f}")
print(f"  Unary + pair: {best_unary_total + best_pair_total:.6f}")

print("\nSelected poses:")
for f in range(1, NUM_FRAGMENTS + 1):
    print(f"  Fragment {f}: case {best_case_sample[f]} -> pose {best_pose_sample[f]}")

pd.DataFrame(best_pair_rows)


Best DQM solution
  Solver: SA
  Energy: 0.103706
  Num occurrences: 1
  Unary total: 0.103706
  Pair total: 0.000000
  Unary + pair: 0.103706

Selected poses:
  Fragment 1: case 12 -> pose 30
  Fragment 2: case 18 -> pose 73
  Fragment 3: case 63 -> pose 76
  Fragment 4: case 40 -> pose 69


,pair,pose_i,pose_j,pair_cost
0,"(1, 2)",30,73,0.0
1,"(1, 3)",30,76,0.0
2,"(1, 4)",30,69,0.0
3,"(2, 3)",73,76,0.0
4,"(2, 4)",73,69,0.0
5,"(3, 4)",76,69,0.0


## Export And Evaluate DQM Solution


In [74]:
# Cell 19 - RDKit helpers for DQM export and evaluation

from rdkit import Chem
from rdkit.Chem import rdFMCS, rdMolAlign, rdShapeHelpers


def fragment_atom_offsets(fragment_mols):
    offsets = {}
    cursor = 0
    for frag_id in sorted(fragment_mols):
        offsets[frag_id] = cursor
        cursor += fragment_mols[frag_id].GetNumAtoms()
    return offsets


def to_global_atom_idx(offsets, fragment_id, atom_idx_1based):
    return offsets[fragment_id] + (atom_idx_1based - 1)


def decap_fragment(mol, fragment_id):
    cap_atoms_1based = FRAGMENT_CAP_ATOMS.get(fragment_id, ())
    if not cap_atoms_1based:
        decapped = Chem.Mol(mol)
        decapped.UpdatePropertyCache(strict=False)
        Chem.FastFindRings(decapped)
        return decapped

    n_atoms = mol.GetNumAtoms()
    to_remove_0based = sorted({idx - 1 for idx in cap_atoms_1based}, reverse=True)
    for idx0 in to_remove_0based:
        if idx0 < 0 or idx0 >= n_atoms:
            raise IndexError(
                f"Cap atom index {idx0 + 1} out of range for fragment {fragment_id} "
                f"with {n_atoms} atoms."
            )

    editable = Chem.RWMol(Chem.Mol(mol))
    for idx0 in to_remove_0based:
        editable.RemoveAtom(idx0)

    decapped = editable.GetMol()
    decapped.UpdatePropertyCache(strict=False)
    Chem.FastFindRings(decapped)
    return decapped


def next_available_sdf_path(output_dir: Path, base_name: str) -> Path:
    copy_index = 1
    while True:
        candidate = output_dir / f"{base_name}_{copy_index}.sdf"
        if not candidate.exists():
            return candidate
        copy_index += 1


def load_mol(path):
    suppl = Chem.SDMolSupplier(str(path), removeHs=False)
    mols = [m for m in suppl if m is not None]
    if not mols:
        raise ValueError(f"Failed to read valid molecule from {path}")
    mol = mols[0]
    if mol.GetNumConformers() == 0:
        raise ValueError(f"Molecule in {path} has no conformer")
    return mol


def heavy_centroid(mol):
    conf = mol.GetConformer()
    pts = []
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() > 1:
            p = conf.GetAtomPosition(atom.GetIdx())
            pts.append([p.x, p.y, p.z])
    pts = np.asarray(pts, dtype=float)
    if len(pts) == 0:
        raise ValueError("No heavy atoms found for centroid computation")
    return pts.mean(axis=0)


def build_heavy_atom_maps(mol1, mol2, timeout=10, max_maps=200):
    mol1_h = Chem.RemoveHs(Chem.Mol(mol1))
    mol2_h = Chem.RemoveHs(Chem.Mol(mol2))

    if mol1_h.GetNumConformers() == 0 or mol2_h.GetNumConformers() == 0:
        raise ValueError("Heavy-atom molecules must both have conformers")

    n1 = mol1_h.GetNumAtoms()
    n2 = mol2_h.GetNumAtoms()
    if n1 != n2:
        raise ValueError(
            f"Heavy atom counts differ: solution={n1}, target={n2}. "
            "These may not be the same ligand."
        )

    mcs = rdFMCS.FindMCS(
        [mol1_h, mol2_h],
        atomCompare=rdFMCS.AtomCompare.CompareElements,
        bondCompare=rdFMCS.BondCompare.CompareOrder,
        ringMatchesRingOnly=True,
        completeRingsOnly=True,
        timeout=timeout,
    )

    if mcs.canceled or mcs.numAtoms == 0:
        raise ValueError("Unable to compute heavy-atom MCS between solution and target")

    if mcs.numAtoms != n1 or mcs.numAtoms != n2:
        raise ValueError(
            f"MCS covers only {mcs.numAtoms} heavy atoms, but solution has {n1} "
            f"and target has {n2}. Full heavy-atom match not found."
        )

    query = Chem.MolFromSmarts(mcs.smartsString)
    if query is None:
        raise ValueError("Failed to build MCS SMARTS query")

    matches1 = mol1_h.GetSubstructMatches(query, uniquify=False)
    matches2 = mol2_h.GetSubstructMatches(query, uniquify=False)

    if not matches1 or not matches2:
        raise ValueError("No heavy-atom MCS matches found between solution and target")

    atom_maps = [list(zip(m1, m2)) for m1 in matches1 for m2 in matches2]
    if not atom_maps:
        raise ValueError("No atom maps could be constructed from MCS matches")

    total_maps = len(atom_maps)
    if total_maps > max_maps:
        atom_maps = atom_maps[:max_maps]

    return mol1_h, mol2_h, atom_maps, total_maps


In [75]:
# Cell 20 - Export a selected DQM solution to final SDF

DQM_SOLUTION_INDEX = 0
POSES_SDF_DIR = Path(f"data/SDFs_{BOX_SIZE}_{DIVISIONS}")

if not POSES_SDF_DIR.exists():
    raise FileNotFoundError(f"Pose SDF directory not found: {POSES_SDF_DIR}")

selected_solution = dqm_solution_frame.iloc[DQM_SOLUTION_INDEX].to_dict()
selected_pose_sample = {
    f: int(selected_solution[f"pose_{f}"])
    for f in range(1, NUM_FRAGMENTS + 1)
}

fragment_suppliers = {}
for frag_id in range(1, NUM_FRAGMENTS + 1):
    frag_path = POSES_SDF_DIR / f"fragment_{frag_id}.sdf"
    mols = [mol for mol in Chem.SDMolSupplier(str(frag_path), removeHs=False) if mol is not None]
    if not mols:
        raise ValueError(f"No valid poses found in {frag_path}")
    fragment_suppliers[frag_id] = mols

raw_mols = {}
for frag_id, pose_idx in selected_pose_sample.items():
    if pose_idx < 0 or pose_idx >= len(fragment_suppliers[frag_id]):
        raise IndexError(
            f"Pose index {pose_idx} is out of range for fragment {frag_id} "
            f"with {len(fragment_suppliers[frag_id])} poses."
        )
    raw_mols[frag_id] = Chem.Mol(fragment_suppliers[frag_id][pose_idx])

selected_mols = {
    frag_id: decap_fragment(mol, frag_id)
    for frag_id, mol in raw_mols.items()
}

offsets = fragment_atom_offsets(selected_mols)

combined = selected_mols[1]
for frag_id in range(2, NUM_FRAGMENTS + 1):
    combined = Chem.CombineMols(combined, selected_mols[frag_id])

editable_mol = Chem.EditableMol(combined)
anchor_indices_to_adjust = set()

for pair in BONDING_FRAGMENTS:
    if pair not in BOND_ANCHORS:
        raise ValueError(
            f"Missing anchor definition for bond pair {pair}. "
            f"Known anchors: {sorted(BOND_ANCHORS.keys())}"
        )
    fa, fb = pair
    aa, ab = BOND_ANCHORS[pair]
    atom_a = to_global_atom_idx(offsets, fa, aa)
    atom_b = to_global_atom_idx(offsets, fb, ab)
    editable_mol.AddBond(atom_a, atom_b, order=Chem.BondType.SINGLE)
    anchor_indices_to_adjust.update([atom_a, atom_b])

combined = editable_mol.GetMol()
for atom_idx in sorted(anchor_indices_to_adjust):
    combined.GetAtomWithIdx(atom_idx).SetNumExplicitHs(0)

Chem.SanitizeMol(combined)

output_dir = Path(f"dqm_results_{RUN_TAG}")
output_dir.mkdir(parents=True, exist_ok=True)

SCORE = f"{float(selected_solution['energy']):.6f}"
base_name = f"{RUN_TAG}_dqm_idx-{DQM_SOLUTION_INDEX}_score-{SCORE}"
SOLUTION_PATH = next_available_sdf_path(output_dir, base_name)

writer = Chem.SDWriter(str(SOLUTION_PATH))
writer.write(combined)
writer.close()

print(f"Exported provisional DQM solution {DQM_SOLUTION_INDEX} to {SOLUTION_PATH}")
print("Selected poses:")
for f in range(1, NUM_FRAGMENTS + 1):
    print(f"  Fragment {f}: pose {selected_pose_sample[f]}")

pd.DataFrame([selected_solution])


Exported provisional DQM solution 0 to dqm_results_SA_14_2/SA_14_2_dqm_idx-0_score-0.103706_1.sdf
Selected poses:
  Fragment 1: pose 30
  Fragment 2: pose 73
  Fragment 3: pose 76
  Fragment 4: pose 69


,solver,energy,num_occurrences,case_1,pose_1,case_2,pose_2,case_3,pose_3,case_4,pose_4
0,SA,0.103706,1,12,30,18,73,63,76,40,69


In [76]:
# Cell 21 - Evaluate exported DQM solution against the target ligand

solution = load_mol(SOLUTION_PATH)
target = load_mol(TARGET_PATH)

solution_heavy, target_heavy, atom_maps, total_maps = build_heavy_atom_maps(solution, target)

solution_copy = Chem.Mol(solution_heavy)
best_rmsd = rdMolAlign.GetBestRMS(solution_copy, target_heavy, map=atom_maps)

# For a single centroid pair, RMSD is the same as Euclidean distance.
centroid_rmsd = float(
    np.linalg.norm(heavy_centroid(solution_heavy) - heavy_centroid(target_heavy))
)

shape_tanimoto_distance = float(rdShapeHelpers.ShapeTanimotoDist(solution_heavy, target_heavy))
shape_tanimoto_score = 1.0 - shape_tanimoto_distance

RMSD_TAG = f"{best_rmsd:.4f}"
final_base_name = (
    f"{SAMPLER}_{BOX_SIZE}_{DIVISIONS}_lmb-{LAMBDA_FRAGMENT}-{LAMBDA_CLASH}-{LAMBDA_BOND}"
    f"_rmsd-{RMSD_TAG}_{SCORE}"
)

if not Path(SOLUTION_PATH).name.startswith(final_base_name):
    final_solution_path = next_available_sdf_path(output_dir, final_base_name)
    Path(SOLUTION_PATH).rename(final_solution_path)
    SOLUTION_PATH = final_solution_path

dqm_metric_summary = pd.DataFrame(
    [
        {
            "solution_index": DQM_SOLUTION_INDEX,
            "solution_path": str(SOLUTION_PATH),
            "target_path": str(TARGET_PATH),
            "score": SCORE,
            "rmsd": float(best_rmsd),
            "centroid_rmsd": centroid_rmsd,
            "tanimoto_score": float(shape_tanimoto_score),
            "tanimoto_distance": float(shape_tanimoto_distance),
            "atom_maps_used": int(len(atom_maps)),
            "atom_maps_total": int(total_maps),
        }
    ]
)

print("=== DQM Solution Evaluation ===")
print(f"Solution index:                    {DQM_SOLUTION_INDEX}")
print(f"Final solution path:               {SOLUTION_PATH}")
print(f"Target path:                       {TARGET_PATH}")
print(f"Heavy atom RMSD:                   {best_rmsd:.4f} A")
print(f"Heavy atom centroid RMSD:          {centroid_rmsd:.4f} A")
print(f"Heavy atom shape Tanimoto score:   {shape_tanimoto_score:.4f}")
print(f"Heavy atom shape Tanimoto dist:    {shape_tanimoto_distance:.4f}")
print(f"Candidate heavy-atom maps used:    {len(atom_maps)} / {total_maps}")

if total_maps > len(atom_maps):
    print(f"WARNING: atom map list was truncated to max_maps={len(atom_maps)}")

print("\n=== Docking Success Criterion ===")
if best_rmsd <= RMSD_CRITERION:
    print(f"SUCCESS (heavy-atom symmetry-aware RMSD <= {RMSD_CRITERION:.1f} A)")
else:
    print(f"FAIL (heavy-atom symmetry-aware RMSD > {RMSD_CRITERION:.1f} A)")

dqm_metric_summary


=== DQM Solution Evaluation ===
Solution index:                    0
Final solution path:               dqm_results_SA_14_2/SA_14_2_lmb-100-0-0_rmsd-2.4749_0.103706_1.sdf
Target path:                       data/target.sdf
Heavy atom RMSD:                   2.4749 A
Heavy atom centroid RMSD:          4.5048 A
Heavy atom shape Tanimoto score:   0.2268
Heavy atom shape Tanimoto dist:    0.7732
Candidate heavy-atom maps used:    36 / 36

=== Docking Success Criterion ===
FAIL (heavy-atom symmetry-aware RMSD > 2.0 A)


,solution_index,solution_path,target_path,score,rmsd,centroid_rmsd,tanimoto_score,tanimoto_distance,atom_maps_used,atom_maps_total
0,0,dqm_results_SA_14_2/SA_14_2_lmb-100-0-0_rmsd-2...,data/target.sdf,0.103706,2.474944,4.504796,0.226843,0.773157,36,36


## Manual Ligand Builder


In [77]:
# Cell 22 - Helper for exporting a manually selected pose combination

def export_pose_sample_to_sdf(pose_sample, output_dir, base_name, poses_sdf_dir=None):
    if poses_sdf_dir is None:
        poses_sdf_dir = Path(f"data/SDFs_{BOX_SIZE}_{DIVISIONS}")

    poses_sdf_dir = Path(poses_sdf_dir)
    if not poses_sdf_dir.exists():
        raise FileNotFoundError(f"Pose SDF directory not found: {poses_sdf_dir}")

    fragment_suppliers = {}
    for frag_id in range(1, NUM_FRAGMENTS + 1):
        frag_path = poses_sdf_dir / f"fragment_{frag_id}.sdf"
        mols = [mol for mol in Chem.SDMolSupplier(str(frag_path), removeHs=False) if mol is not None]
        if not mols:
            raise ValueError(f"No valid poses found in {frag_path}")
        fragment_suppliers[frag_id] = mols

    raw_mols = {}
    for frag_id in range(1, NUM_FRAGMENTS + 1):
        pose_idx = int(pose_sample[frag_id])
        if pose_idx < 0 or pose_idx >= len(fragment_suppliers[frag_id]):
            raise IndexError(
                f"Pose index {pose_idx} is out of range for fragment {frag_id} "
                f"with {len(fragment_suppliers[frag_id])} poses."
            )
        raw_mols[frag_id] = Chem.Mol(fragment_suppliers[frag_id][pose_idx])

    selected_mols = {
        frag_id: decap_fragment(mol, frag_id)
        for frag_id, mol in raw_mols.items()
    }

    offsets = fragment_atom_offsets(selected_mols)

    combined = selected_mols[1]
    for frag_id in range(2, NUM_FRAGMENTS + 1):
        combined = Chem.CombineMols(combined, selected_mols[frag_id])

    editable_mol = Chem.EditableMol(combined)
    anchor_indices_to_adjust = set()

    for pair in BONDING_FRAGMENTS:
        if pair not in BOND_ANCHORS:
            raise ValueError(
                f"Missing anchor definition for bond pair {pair}. "
                f"Known anchors: {sorted(BOND_ANCHORS.keys())}"
            )
        fa, fb = pair
        aa, ab = BOND_ANCHORS[pair]
        atom_a = to_global_atom_idx(offsets, fa, aa)
        atom_b = to_global_atom_idx(offsets, fb, ab)
        editable_mol.AddBond(atom_a, atom_b, order=Chem.BondType.SINGLE)
        anchor_indices_to_adjust.update([atom_a, atom_b])

    combined = editable_mol.GetMol()
    for atom_idx in sorted(anchor_indices_to_adjust):
        combined.GetAtomWithIdx(atom_idx).SetNumExplicitHs(0)

    Chem.SanitizeMol(combined)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    solution_path = next_available_sdf_path(output_dir, base_name)

    writer = Chem.SDWriter(str(solution_path))
    writer.write(combined)
    writer.close()

    return solution_path


In [78]:
# Cell 23 - Pick poses manually for each fragment

AVAILABLE_MANUAL_POSES = {
    frag_id: sorted(fragments[frag_id]["p"].astype(int).tolist())
    for frag_id in range(1, NUM_FRAGMENTS + 1)
}

for frag_id in range(1, NUM_FRAGMENTS + 1):
    poses = AVAILABLE_MANUAL_POSES[frag_id]
    preview = poses[:15]
    suffix = " ..." if len(poses) > 15 else ""
    print(f"Fragment {frag_id}: {len(poses)} available pruned poses -> {preview}{suffix}")

MANUAL_POSE_SAMPLE = {
    # Edit these p values directly.
    1: 79,
    2: 91,
    3: 169,
    4: 120,
}

MANUAL_EXPORT_DIR = Path(f"manual_ligands_{BOX_SIZE}_{DIVISIONS}")

for frag_id in range(1, NUM_FRAGMENTS + 1):
    if frag_id not in MANUAL_POSE_SAMPLE:
        raise KeyError(f"Missing manual pose selection for fragment {frag_id}")

manual_pose_summary = pd.DataFrame(
    [
        {
            "fragment": frag_id,
            "pose": int(MANUAL_POSE_SAMPLE[frag_id]),
            "present_after_pruning": int(MANUAL_POSE_SAMPLE[frag_id]) in AVAILABLE_MANUAL_POSES[frag_id],
        }
        for frag_id in range(1, NUM_FRAGMENTS + 1)
    ]
)

manual_pose_summary


Fragment 1: 111 available pruned poses -> [0, 1, 3, 6, 7, 8, 10, 12, 13, 15, 16, 25, 30, 37, 42] ...
Fragment 2: 91 available pruned poses -> [0, 1, 2, 3, 4, 10, 12, 13, 16, 22, 29, 33, 40, 46, 48] ...
Fragment 3: 302 available pruned poses -> [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 13, 15, 16, 17, 18] ...
Fragment 4: 230 available pruned poses -> [0, 1, 2, 3, 4, 7, 12, 14, 18, 19, 20, 21, 23, 24, 25] ...


,fragment,pose,present_after_pruning
0,1,79,True
1,2,91,True
2,3,169,True
3,4,120,True


In [79]:
# Cell 24 - Export the manually assembled ligand to SDF

manual_base_name = (
    f"manual_{BOX_SIZE}_{DIVISIONS}_"
    + "_".join(
        f"p{frag_id}-{int(MANUAL_POSE_SAMPLE[frag_id])}"
        for frag_id in range(1, NUM_FRAGMENTS + 1)
    )
)

MANUAL_SOLUTION_PATH = export_pose_sample_to_sdf(
    MANUAL_POSE_SAMPLE,
    MANUAL_EXPORT_DIR,
    manual_base_name,
)

print(f"Exported manual ligand to {MANUAL_SOLUTION_PATH}")
print("Manual pose selection:")
for frag_id in range(1, NUM_FRAGMENTS + 1):
    print(f"  Fragment {frag_id}: pose {int(MANUAL_POSE_SAMPLE[frag_id])}")


Exported manual ligand to manual_ligands_14_2/manual_14_2_p1-79_p2-91_p3-169_p4-120_2.sdf
Manual pose selection:
  Fragment 1: pose 79
  Fragment 2: pose 91
  Fragment 3: pose 169
  Fragment 4: pose 120


In [80]:
# Cell 25 - Evaluate the manually assembled ligand against the target

manual_solution = load_mol(MANUAL_SOLUTION_PATH)
manual_target = load_mol(TARGET_PATH)

manual_solution_heavy, manual_target_heavy, manual_atom_maps, manual_total_maps = build_heavy_atom_maps(
    manual_solution,
    manual_target,
)

manual_solution_copy = Chem.Mol(manual_solution_heavy)
manual_best_rmsd = rdMolAlign.GetBestRMS(
    manual_solution_copy,
    manual_target_heavy,
    map=manual_atom_maps,
)

manual_centroid_rmsd = float(
    np.linalg.norm(
        heavy_centroid(manual_solution_heavy) - heavy_centroid(manual_target_heavy)
    )
)

manual_tanimoto_distance = float(
    rdShapeHelpers.ShapeTanimotoDist(manual_solution_heavy, manual_target_heavy)
)
manual_tanimoto_score = 1.0 - manual_tanimoto_distance

manual_metric_summary = pd.DataFrame(
    [
        {
            "solution_path": str(MANUAL_SOLUTION_PATH),
            "target_path": str(TARGET_PATH),
            "rmsd": float(manual_best_rmsd),
            "centroid_rmsd": manual_centroid_rmsd,
            "tanimoto_score": float(manual_tanimoto_score),
            "tanimoto_distance": float(manual_tanimoto_distance),
            "atom_maps_used": int(len(manual_atom_maps)),
            "atom_maps_total": int(manual_total_maps),
        }
    ]
)

print("=== Manual Ligand Evaluation ===")
print(f"Manual solution path:              {MANUAL_SOLUTION_PATH}")
print(f"Target path:                       {TARGET_PATH}")
print(f"Heavy atom RMSD:                   {manual_best_rmsd:.4f} A")
print(f"Heavy atom centroid RMSD:          {manual_centroid_rmsd:.4f} A")
print(f"Heavy atom shape Tanimoto score:   {manual_tanimoto_score:.4f}")
print(f"Heavy atom shape Tanimoto dist:    {manual_tanimoto_distance:.4f}")
print(f"Candidate heavy-atom maps used:    {len(manual_atom_maps)} / {manual_total_maps}")

if manual_total_maps > len(manual_atom_maps):
    print(f"WARNING: atom map list was truncated to max_maps={len(manual_atom_maps)}")

manual_metric_summary


=== Manual Ligand Evaluation ===
Manual solution path:              manual_ligands_14_2/manual_14_2_p1-79_p2-91_p3-169_p4-120_2.sdf
Target path:                       data/target.sdf
Heavy atom RMSD:                   0.5597 A
Heavy atom centroid RMSD:          0.5044 A
Heavy atom shape Tanimoto score:   0.7314
Heavy atom shape Tanimoto dist:    0.2686
Candidate heavy-atom maps used:    36 / 36


,solution_path,target_path,rmsd,centroid_rmsd,tanimoto_score,tanimoto_distance,atom_maps_used,atom_maps_total
0,manual_ligands_14_2/manual_14_2_p1-79_p2-91_p3...,data/target.sdf,0.559723,0.50437,0.7314,0.2686,36,36
